# CI RAG Pipeline — Advanced Modular RAG with PubMedBERT + FAISS + BM25

In [ ]:
import pathlib
import sys
import os

import pandas as pd
from IPython.display import display, Markdown

_here = pathlib.Path().resolve()
if (_here / 'clinical-trial-optimizer').exists():
    CT_ROOT = _here / 'clinical-trial-optimizer'
elif (_here / 'src').exists():
    CT_ROOT = _here
elif (_here.parent / 'src').exists():
    CT_ROOT = _here.parent
else:
    raise RuntimeError(f'Cannot locate clinical-trial-optimizer root from {_here}')

sys.path.insert(0, str(CT_ROOT / 'src'))

from ct_api import fetch_competing_trials
from rag.chunker import chunk_competing_trials
from rag.embedder import load_pubmedbert, embed_chunks
from rag.index import build_index
from rag.retriever import retrieve
from rag.ci_reasoner import reason_about_criterion
from rag.recommendation_evaluator import evaluate_recommendations

print(f'CT_ROOT : {CT_ROOT}')

In [ ]:
print('API key set' if os.environ.get('ANTHROPIC_API_KEY') else 'WARNING: ANTHROPIC_API_KEY not set')

In [ ]:
# Paste I/E criteria here
ie_criteria = ""


In [ ]:
# ⚠️ HUMAN APPROVAL REQUIRED — this cell makes a live API call to ClinicalTrials.gov
competing_trials_md = fetch_competing_trials(ie_criteria)
print(f'Trials retrieved: {competing_trials_md.count("NCT")}')

In [ ]:
# ⚠️ HUMAN APPROVAL REQUIRED — this cell calls Claude Haiku to extract criteria
competing_chunks = chunk_competing_trials(competing_trials_md)
print(f'Criteria chunks extracted: {len(competing_chunks)}')

In [ ]:
# Load PubMedBERT and embed all chunks
model = load_pubmedbert()
embeddings = embed_chunks(competing_chunks, model)
print(f'Embeddings shape: {embeddings.shape}')

# Build hybrid index (FAISS + BM25)
PERSIST_DIR = CT_ROOT / 'data' / 'indexes'
index = build_index(competing_chunks, embeddings, collection_name='ci_trials', persist_dir=str(PERSIST_DIR))
print(f'Index built: {len(index.chunks)} chunks indexed')

In [ ]:
# For each source criterion, retrieve relevant competing trial criteria
# TODO: parse source I/E criteria and retrieve for each
# source_criteria = parse_criteria_via_llm(ie_criteria, "SOURCE", "My Trial", {})
# for crit in source_criteria:
#     results = retrieve(crit.text, index, model, k=10)
#     print(f"Retrieved {len(results)} chunks for: {crit.text[:60]}...")

In [ ]:
# ⚠️ HUMAN APPROVAL REQUIRED — this cell calls Claude Haiku for each criterion
# For each source criterion and its retrieved results, reason about KEEP/RELAX/TIGHTEN
# recommendations = []
# for crit in source_criteria:
#     results = retrieve(crit.text, index, model, k=10)
#     rec = reason_about_criterion(crit.text, results)
#     recommendations.append(rec)
# print(f"Recommendations generated: {len(recommendations)}")

In [ ]:
# Evaluate recommendations and display report
# report = evaluate_recommendations(recommendations)
# print(f"\n=== CI RAG EVALUATION REPORT ===")
# print(f"Total criteria: {report.total}")
# print(f"KEEP: {report.keep_count}, RELAX: {report.relax_count}, TIGHTEN: {report.tighten_count}")
# print(f"Avg confidence: {report.avg_confidence:.2f}")
# print(f"Consistency score: {report.consistency_score:.2f}")